In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [2]:
import torch
import numpy as np
import transformers

In [3]:
from transformers import (
    HfArgumentParser,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    Wav2Vec2Config,
    is_apex_available,
    set_seed,
)

/home/ubuntu/.local/lib/python3.8/site-packages/apex/pyprof/__init__.py:5: FutureWarning: pyprof will be removed by the end of June, 2022
  warnings.warn("pyprof will be removed by the end of June, 2022", FutureWarning)


In [4]:
import string
import json

CTC_VOCAB = [''] + list(string.ascii_lowercase + string.digits) + [' ']

In [5]:
vocab_dict = {v: k for k, v in enumerate(CTC_VOCAB)}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("ctc-vocab.json", "w") as vocab_file:
    json.dump(vocab_dict, vocab_file)

tokenizer = Wav2Vec2CTCTokenizer(
    "ctc-vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [6]:
model = Wav2Vec2ForCTC.from_pretrained(
    './mixed-300m-half',
    ctc_loss_reduction="mean",
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(tokenizer),
)

In [ ]:
model

In [7]:
half_config = Wav2Vec2Config.from_pretrained('malay-huggingface/wav2vec2-xls-r-300m-mixed', 
                                             num_hidden_layers = 8,
                                            ctc_loss_reduction="mean",
                                             pad_token_id=tokenizer.pad_token_id,
                                             vocab_size=len(tokenizer))

In [8]:
half = Wav2Vec2ForCTC(half_config)

In [9]:
import torch.nn as nn

def copy_layers(src_layers, dest_layers, layers_to_copy):
    layers_to_copy = nn.ModuleList([src_layers[i] for i in layers_to_copy])
    assert len(dest_layers) == len(layers_to_copy), f"{len(dest_layers)} != {len(layers_to_copy)}"
    dest_layers.load_state_dict(layers_to_copy.state_dict())

layers_to_copy = [0,1,2,3,20,21,22,23]

copy_layers(model.wav2vec2.encoder.layers, half.wav2vec2.encoder.layers, layers_to_copy)

In [10]:
half.save_pretrained("mixed-300m-half")

In [11]:
half

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (2): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (3): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elemen

In [12]:
model

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (2): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (3): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elemen

In [7]:
model.wav2vec2.encoder.layers[0].state_dict()

OrderedDict([('attention.k_proj.weight',
              tensor([[ 0.0229, -0.0140,  0.0782,  ...,  0.0366,  0.0898,  0.0535],
                      [-0.0626, -0.0029, -0.0439,  ...,  0.0067, -0.0536, -0.1597],
                      [-0.0933,  0.0489,  0.0332,  ...,  0.0175, -0.1014, -0.0816],
                      ...,
                      [-0.0392,  0.0120,  0.0791,  ..., -0.0088,  0.0159,  0.0011],
                      [-0.0361, -0.0270, -0.0319,  ..., -0.0316, -0.0034, -0.0142],
                      [-0.0009, -0.0011, -0.0117,  ...,  0.0103,  0.0102, -0.0183]])),
             ('attention.k_proj.bias',
              tensor([ 0.0172, -0.0070, -0.0315,  ..., -0.0132,  0.0085,  0.0360])),
             ('attention.v_proj.weight',
              tensor([[-0.0256, -0.0152, -0.0873,  ..., -0.0421,  0.0454,  0.1536],
                      [-0.0077,  0.0171,  0.1195,  ..., -0.0231, -0.0445, -0.0151],
                      [ 0.0624, -0.0047, -0.1042,  ...,  0.0066, -0.0102, -0.0368],
        

In [16]:
half.wav2vec2.encoder.layers[0].state_dict()

OrderedDict([('attention.k_proj.weight',
              tensor([[ 0.0229, -0.0140,  0.0782,  ...,  0.0366,  0.0898,  0.0535],
                      [-0.0626, -0.0029, -0.0439,  ...,  0.0067, -0.0536, -0.1597],
                      [-0.0933,  0.0489,  0.0332,  ...,  0.0175, -0.1014, -0.0816],
                      ...,
                      [-0.0392,  0.0120,  0.0791,  ..., -0.0088,  0.0159,  0.0011],
                      [-0.0361, -0.0270, -0.0319,  ..., -0.0316, -0.0034, -0.0142],
                      [-0.0009, -0.0011, -0.0117,  ...,  0.0103,  0.0102, -0.0183]])),
             ('attention.k_proj.bias',
              tensor([ 0.0172, -0.0070, -0.0315,  ..., -0.0132,  0.0085,  0.0360])),
             ('attention.v_proj.weight',
              tensor([[-0.0256, -0.0152, -0.0873,  ..., -0.0421,  0.0454,  0.1536],
                      [-0.0077,  0.0171,  0.1195,  ..., -0.0231, -0.0445, -0.0151],
                      [ 0.0624, -0.0047, -0.1042,  ...,  0.0066, -0.0102, -0.0368],
        